In [ ]:
import copy
from server.fedavg import fedavg


def run_federated(model, clients, selection_fn, rounds, k):
    """
    Standard federated training with a generic selection function.
    selection_fn(client_ids, k) -> list of selected IDs
    """
    global_model = model

    for r in range(rounds):
        print(f"Round {r+1} completed")

        # Select clients
        selected = selection_fn(list(clients.keys()), k)

        # Get gradient updates from selected clients only
        updates = []
        for c in selected:
            updates.append(clients[c].get_update(global_model))

        global_model = fedavg(global_model, updates)

    return global_model


def run_federated_divfl(model, clients, selection_fn, rounds, k):
    """
    Federated training with DivFL-style selection.
    Collects ALL client updates, then selects diverse subset.
    """
    global_model = model

    for r in range(rounds):
        print(f"Round {r+1}")

        # Step 1: get updates from ALL clients
        client_updates = {}
        for c in clients:
            client_updates[c] = clients[c].get_update(global_model)

        # Step 2: select diverse clients
        selected = selection_fn(client_updates, k)

        # Step 3: aggregate ONLY selected updates
        updates = [client_updates[c] for c in selected]

        global_model = fedavg(global_model, updates)

    return global_model


def run_sfedavg(model, clients, val_loader, rounds, k, T=10):
    """
    S-FedAvg: Shapley Value based Federated Averaging.

    Implements the S-FedAvg algorithm from:
      'Game of Gradients: Mitigating Irrelevant Clients in Federated Learning'
      Nagalapatti & Narayanam, AAAI 2021

    Each round:
      1. All clients compute gradient updates
      2. Server computes Shapley value for each client (via validation accuracy)
      3. Softmax(Shapley scores) used to probabilistically select k clients
      4. Selected clients' updates are aggregated via FedAvg

    Args:
        model     : Initial global model
        clients   : dict {client_id: Client}
        val_loader: DataLoader for the server's validation set
        rounds    : Number of communication rounds
        k         : Number of clients to select per round
        T         : Monte Carlo permutation samples for Shapley estimation

    Returns:
        global_model: Trained global model
    """
    import torch
    from selection.shapley import compute_shapley_values, sfedavg_selection

    device = next(model.parameters()).device
    global_model = model

    for r in range(rounds):
        # Step 1: Collect gradient updates from ALL clients
        client_updates = {}
        for c in clients:
            client_updates[c] = clients[c].get_update(global_model)

        # Step 2: Compute Shapley values using validation set
        shapley_scores = compute_shapley_values(
            global_model, client_updates, val_loader, device, T=T
        )

        # Step 3: Softmax-based probabilistic client selection
        selected = sfedavg_selection(shapley_scores, k)

        # Step 4: Aggregate selected updates via FedAvg
        updates = [client_updates[c] for c in selected]
        global_model = fedavg(global_model, updates)

        # Log round summary
        scores_str = {c: f"{shapley_scores[c]:.4f}" for c in shapley_scores}
        print(f"Round {r+1:02d} | Shapley scores: {scores_str} | Selected: {selected}")

    return global_model
